# MelanCare: train and evaluate the baseline model

Runs the whole pipeline on a free Colab GPU: data download, preparation, training, evaluation and
Grad-CAM. Takes about 30-45 minutes.

1. *Runtime → Change runtime type → **T4 GPU** → Save*
2. *Runtime → **Run all***
3. Keep this tab open. At the end, the results zip downloads to your computer.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime → Change runtime type → T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Get the code

In [ ]:
import os
os.chdir("/content")
!rm -rf melancare && git clone -q https://github.com/i-oladele/melancare.git
os.chdir("/content/melancare")
!pip install -q -e .

## 2. Download the data (about 3.2 GB)

In [ ]:
%%bash
mkdir -p data/raw && cd data/raw
B=https://dataverse.harvard.edu/api/access/datafile
S=https://isic-archive.s3.amazonaws.com/challenges/2018
curl -sSL --retry 5 -o HAM10000_metadata.csv "$B/4338392?format=original"
curl -sSL --retry 5 -o ISIC2018_Task3_Test_GroundTruth.csv "$B/6924466?format=original"
curl -sSL --retry 5 -o HAM10000_segmentations.zip "$B/3838943"
curl -sSL --retry 5 -o ISIC2018_Task3_Training_Input.zip "$S/ISIC2018_Task3_Training_Input.zip"
curl -sSL --retry 5 -o ISIC2018_Task3_Test_Input.zip "$S/ISIC2018_Task3_Test_Input.zip"
ls -lh

## 3. Resize images and create lesion-level splits

In [ ]:
!python scripts/prepare_data.py

## 4. Train

In [ ]:
!python scripts/train.py --run-name effb0_weighted --num-workers 2

## 5. Evaluate and explain

In [ ]:
!python scripts/evaluate.py --run runs/effb0_weighted --num-workers 2
!python scripts/explain.py --run runs/effb0_weighted

In [ ]:
from IPython.display import Image, Markdown, display
run = "runs/effb0_weighted"
display(Image(f"{run}/training_curves.png", width=800))
display(Markdown(open(f"{run}/evaluation/report.md").read()))
for f in ["confusion_test.png", "roc_malignant_test.png", "confusion_external.png"]:
    display(Image(f"{run}/evaluation/{f}", width=550))
display(Markdown(open(f"{run}/explain/summary.md").read()))
for f in ["gradcam_correct_by_class.png", "gradcam_confident_errors.png", "gradcam_least_lesion_focus.png"]:
    display(Image(f"{run}/explain/{f}", width=900))

## Save the results
Colab deletes everything when the session ends. If the download doesn't start, open the **Files**
panel (folder icon on the left), find `effb0_weighted_results.zip` in `melancare`, and choose **⋮ → Download**.

In [ ]:
!cd runs && zip -qr ../effb0_weighted_results.zip effb0_weighted
from google.colab import files
files.download("effb0_weighted_results.zip")